In [ ]:
import os
import shutil

# Base folder for assignment
BASE = "/content/drive/MyDrive/Assignment2/Used_Cars"
os.makedirs(BASE, exist_ok=True)
os.makedirs(os.path.join(BASE,"notebooks"), exist_ok=True)
os.makedirs(os.path.join(BASE,"data"), exist_ok=True)
os.makedirs(os.path.join(BASE,"reports"), exist_ok=True)
os.makedirs(os.path.join(BASE,"outputs"), exist_ok=True)

print("Folders created:", BASE)

Folders created: /content/drive/MyDrive/Assignment2/Used_Cars


In [ ]:
import shutil
import os

# Assuming BASE is already defined, e.g.:
# BASE = "/content/drive/MyDrive/Assignment2/Used_Cars"

shutil.copy("/content/train.csv", os.path.join(BASE, "data", "train.csv"))
print("Dataset copied to data/ folder")

Dataset copied to data/ folder


Data Preprocessing & Handling Missing Values

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load dataset
df = pd.read_csv(os.path.join(BASE,"data","train.csv"))

# Function to extract numeric values
def extract_number(value):
    try:
        return float(str(value).split()[0].replace(',', ''))
    except:
        return np.nan

# Clean numeric columns
for col in ['Mileage','Engine','Power']:
    df[col] = df[col].apply(extract_number)

# Clean New_Price
df['New_Price'] = df['New_Price'].astype(str).str.replace('Lakh|Cr','', regex=True)
df['New_Price'] = pd.to_numeric(df['New_Price'], errors='coerce')

# Impute numeric columns with median
num_cols = ['Mileage','Engine','Power','Seats','New_Price']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Impute categorical columns with mode
cat_cols = ['Location','Fuel_Type','Transmission']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Check missing values after handling
print("Missing values after handling:\n", df.isnull().sum())

Missing values after handling:
 Unnamed: 0           0
Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
New_Price            0
Price                0
dtype: int64


Median used for numeric columns → handles outliers better than mean.
Mode used for categorical columns → preserves most common value in each category.

One-hot Encoding

In [ ]:
df = pd.get_dummies(df, columns=['Fuel_Type','Transmission'], drop_first=False)
print("One-hot encoded columns:", [col for col in df.columns if 'Fuel_Type' in col or 'Transmission' in col])

One-hot encoded columns: ['Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual']


In [ ]:
df['Car_Age'] = datetime.now().year - df['Year']
df[['Year','Car_Age']].head()

,Year,Car_Age
0,2015,10
1,2011,14
2,2012,13
3,2013,12
4,2013,12


In [ ]:
# Select & Rename
selected_df = df[['Name','Location','Year','Mileage','Engine','Power','Price']].copy()
selected_df.rename(columns={'Name':'Car_Name'}, inplace=True)

# Filter: Cars priced above 10 lakhs
filtered_df = selected_df[selected_df['Price'] > 10]

# Mutate: New feature km_per_cc
df['km_per_cc'] = df['Mileage'] / df['Engine']

# Arrange: sort by Price descending
arranged_df = df.sort_values(by='Price', ascending=False)

# Summarize: group by Fuel_Type_Petrol
fuel_column = 'Fuel_Type_Petrol'
summary = df.groupby(fuel_column).agg({
    'Price':['mean','max'],
    'Mileage':'mean',
    'Power':'mean'
}).round(2)

# Print outputs
print("Selected & Renamed Columns:\n", selected_df.head())
print("\nCars priced above 10 lakhs:\n", filtered_df.head())
print("\nNew column 'km_per_cc':\n", df[['Mileage','Engine','km_per_cc']].head())
print("\nTop 5 most expensive cars:\n", arranged_df[['Name','Price']].head())
print("\nGrouped summary by fuel type:\n", summary)

Selected & Renamed Columns:
                            Car_Name    Location  Year  Mileage  Engine  \
0  Hyundai Creta 1.6 CRDi SX Option        Pune  2015    19.67  1582.0   
1                      Honda Jazz V     Chennai  2011    13.00  1199.0   
2                 Maruti Ertiga VDI     Chennai  2012    20.77  1248.0   
3   Audi A4 New 2.0 TDI Multitronic  Coimbatore  2013    15.20  1968.0   
4            Nissan Micra Diesel XV      Jaipur  2013    23.08  1461.0   

    Power  Price  
0  126.20  12.50  
1   88.70   4.50  
2   88.76   6.00  
3  140.80  17.74  
4   63.10   3.50  

Cars priced above 10 lakhs:
                              Car_Name    Location  Year  Mileage  Engine  \
0    Hyundai Creta 1.6 CRDi SX Option        Pune  2015    19.67  1582.0   
3     Audi A4 New 2.0 TDI Multitronic  Coimbatore  2013    15.20  1968.0   
5   Toyota Innova Crysta 2.8 GX AT 8S      Mumbai  2016    11.36  2755.0   
11   Land Rover Range Rover 2.2L Pure       Delhi  2014    12.70  2179.0   
12

In [ ]:
findings_text = """# Q1 Used Cars Analysis Findings

## Part (a) — Missing Values
- Numeric columns filled with median to handle outliers and skewed distributions.
- Categorical columns filled with mode to preserve the most common value.

## Part (b) — Numeric Conversion
- Units removed from Mileage, Engine, Power, and New_Price.

## Part (c) — One-Hot Encoding
- Fuel_Type and Transmission converted to numeric columns.

## Part (d) — Car Age Feature
- Added Car_Age = Current Year - Year.

## Part (e) — Data Manipulations
- Selected & renamed columns.
- Filtered cars priced > 10 lakhs.
- Created km_per_cc column.
- Sorted by Price.
- Grouped by Fuel_Type_Petrol with summary statistics.
"""

with open(os.path.join(BASE,"reports","findings.md"), "w") as f:
    f.write(findings_text)

what_text = """Steps performed in Used Cars Analysis (Q1):
1. Created assignment folders in Google Drive.
2. Uploaded dataset to data/ folder.
3. Handled missing values (median/mode).
4. Cleaned numeric columns (removed units).
5. One-hot encoded categorical columns.
6. Created Car_Age feature.
7. Performed select, filter, rename, mutate, arrange, groupby operations.
8. Saved findings.md and what_I_have_done.txt in reports/ folder.
"""

with open(os.path.join(BASE,"reports","what_I_have_done.txt"), "w") as f:
    f.write(what_text)

print("Reports saved successfully.")

Reports saved successfully.


In [ ]:
zip_out = "/content/drive/MyDrive/Assignment2/Used_Cars_Final.zip"
shutil.make_archive(zip_out.replace(".zip",""), 'zip', BASE)
print("Created zip:", zip_out)

Created zip: /content/drive/MyDrive/Assignment2/Used_Cars_Final.zip


In [12]:
from google.colab import files

files.download(zip_out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>